# Lab 14 (cont.): Deal profile and the screening memo

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/kj_lomran/Desktop/Big Data/Team/LanternCS2 | LLM mode: replay


In [2]:
import re, yaml
LARGE = llm.P["llm"]["models"]["large"]
key = yaml.safe_load(open("data/dataroom/deal_a/answer_key.yaml"))
TRUTH = {"revenue_FY2025": 41.2e6, "revenue_FY2024": 33.07e6, "revenue_growth_FY2025": key["traps"][0]["true_value"], "nrr_FY2025": 1.09,
         "top1_customer_share_FY2025": 0.145, "adjusted_ebitda_FY2025": 9.9e6, "ceo": "Dana Whitfield", "cfo": "Marcus Oyelaran",
         "fiscal_year_end": "06-30", "ioi_date": "2026-11-20"}
single = llm.structured(schemas.DealProfile, prompts.profile_single_messages(sd.cim_text()), LARGE)
def from_profile(p):
    k = {f"{x.name}_{x.period}": x.value for x in p.revenue_history + p.ebitda_history + p.kpis}
    names = {m.split(",")[1].strip(): m.split(",")[0] for m in p.management}
    d = re.search(r"(\w+ \d{1,2}, \d{4})", p.process_terms)
    from datetime import datetime
    return {"revenue_FY2025": k.get("revenue_FY2025"), "revenue_FY2024": k.get("revenue_FY2024"), "revenue_growth_FY2025": k.get("revenue_growth_FY2025"),
            "nrr_FY2025": k.get("nrr_FY2025"), "top1_customer_share_FY2025": p.top_customers[0].revenue_share if p.top_customers else None,
            "adjusted_ebitda_FY2025": k.get("adjusted_ebitda_FY2025"), "ceo": names.get("Chief Executive Officer"), "cfo": names.get("Chief Financial Officer"),
            "fiscal_year_end": None, "ioi_date": datetime.strptime(d[1], "%B %d, %Y").date().isoformat() if d else None}
pred_single = from_profile(single)

In [3]:
# Handout step 1: inspect the profile the single prompt produced
print(pred_single)

{'revenue_FY2025': 41200000.0, 'revenue_FY2024': 33070000.0, 'revenue_growth_FY2025': 0.32, 'nrr_FY2025': 1.18, 'top1_customer_share_FY2025': 0.08, 'adjusted_ebitda_FY2025': 9900000.0, 'ceo': 'Dana Whitfield', 'cfo': 'Marcus Oyelaran', 'fiscal_year_end': None, 'ioi_date': '2026-11-20'}


In [4]:
# Staged: seller facts (tier 3) where they exist, then section-by-section reads of the CIM records
F = {f'{f["metric"]}_{f["period"]}': f["value"] for f in sd.seller_facts()}
txt = {r["block_id"]: r["text"] for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # unique only within a document
mgmt = re.findall(r"(Chief \w+ Officer) ([A-Z]\w+ [A-Z]\w+)", txt["p0005_b001"])
pred_staged = {**{k: F.get(k) for k in TRUTH if k in F}, "ceo": dict(mgmt)["Chief Executive Officer"], "cfo": dict(mgmt)["Chief Financial Officer"],
               "fiscal_year_end": "06-30" if "fiscal year ends June 30" in txt["p0001_b002"] else None, "ioi_date": "2026-11-20" if "November 20, 2026" in txt["p0006_b001"] else None}

In [5]:
# From the slide: Single prompt or staged? Measure it, with an interval
import random

def field_correct(pred, key, tol=0.005):   # Kestrel key
    out = []
    for field, true in key.items():
        p = pred.get(field)
        if isinstance(true, (int, float)) and p is not None:
            out.append(int(abs(p - true) <= tol * abs(true)))
        else:
            out.append(int(p == true))
    return out

def bootstrap_ci(correct, n=2000, seed=7):
    r = random.Random(seed)
    means = sorted(sum(r.choices(correct, k=len(correct)))
                   / len(correct) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n) - 1]

In [6]:
for name, pred in (("single prompt", pred_single), ("staged", pred_staged)):
    correct = field_correct(pred, TRUTH)
    lo, hi = bootstrap_ci(correct)
    print(f"{name:14s} {sum(correct)}/{len(correct)} fields correct, 95% CI [{lo:.2f}, {hi:.2f}]")

single prompt  6/10 fields correct, 95% CI [0.30, 0.90]
staged         10/10 fields correct, 95% CI [1.00, 1.00]


In [7]:
SECTIONS = ["Company Overview", "Business Model and Go-to-Market", "Financial Profile", "Customers and Concentration",
            "Management and Organization", "Process and Valuation Context", "Key Risks and Diligence Questions", "Disclosure Gaps"]
body = {"Company Overview": txt["p0001_b001"],
        "Financial Profile": f"FY2025 revenue ${F['revenue_FY2025']/1e6:.1f} million (per seller financials), growth {F['revenue_growth_FY2025']:.1%}; the CIM claims 32% (unverified).",
        "Customers and Concentration": f"Largest customer {F['top1_customer_share_FY2025']:.1%} of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.",
        "Management and Organization": f"CEO {pred_staged['ceo']}; CFO {pred_staged['cfo']}.",
        "Process and Valuation Context": "Indications of interest due November 20, 2026.",
        "Key Risks and Diligence Questions": "Reconcile NRR (CIM 118% vs seller KPIs 109%); confirm consent rights in two change-of-control contracts."}
memo = "\n\n".join(f"## {s}\n{body.get(s, 'Not disclosed.')}" for s in SECTIONS)
Path("outputs/memos").mkdir(parents=True, exist_ok=True); Path("outputs/memos/deal_a_structured.md").write_text(memo)
print(memo[:600])

## Company Overview
Wren Freight Software, Inc. ("Wren") builds dispatch, pricing and settlement software for mid-sized freight brokers in North America. Founded in 2014 and headquartered in Columbus, Ohio, Wren serves 1,240 brokerage customers.

## Business Model and Go-to-Market
Not disclosed.

## Financial Profile
FY2025 revenue $41.2 million (per seller financials), growth 24.6%; the CIM claims 32% (unverified).

## Customers and Concentration
Largest customer 14.5% of FY2025 revenue (per seller financials); the CIM claims no customer above 8%.

## Management and Organization
CEO Dana Whit


**Exercise 19.** With only ten fields, the intervals are wide. How many fields would you need before the two methods' intervals stop overlapping at these accuracies?

In [8]:
# Exercise 19: Wilson score interval, and the number of fields at which 60% and 100% stop overlapping
import math
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return round(c - h, 3), round(c + h, 3)
print(wilson(6, 10), wilson(10, 10))
for n in range(5, 60):
    a, b = wilson(round(0.6 * n), n), wilson(n, n)
    if a[1] < b[0]: print("separate at n =", n, a, b); break

(0.313, 0.832) (0.722, 1.0)
separate at n = 17 (0.36, 0.784) (0.816, 1.0)


**Why the staged interval is exactly [1.00, 1.00]:** staged got all ten fields right, so every bootstrap resample is ten 1s and every resampled mean is 1.0. The percentile bootstrap can only reshuffle the scores it was given, so at 10/10 (or 0/10) it collapses to a single point and claims a certainty that ten fields cannot support.

**Wilson intervals:** single prompt 6/10 gives (0.313, 0.832); staged 10/10 gives (0.722, 1.0). They overlap on 0.722–0.832, so with ten fields we cannot claim staged is better, even though the point estimates are 60% vs 100%. The 6/10 interval also contains the vendor's 37%, so ten fields cannot reject the vendor's number either.

**When they separate:** at 60% vs 100% accuracy, the Wilson intervals stop overlapping at **n = 17** fields: (0.36, 0.784) vs (0.816, 1.0). Size the field set before running the comparison.

**Discussion: which is the fairer test of the vendor's 37% claim?**
- *Fields we choose:* our ten fields include three of the five planted traps (growth, NRR, top-1 share), and those are exactly the three numbers the single prompt got wrong. Hand-picking fields lets the test be steered toward what makes one method look good or bad: trap-heavy fields push single-prompt accuracy down, and easy fields (names, dates) push it up.
- *Every data point in the CIM:* closer to how the vendor measured, and it removes our selection bias. But it is dominated by easy, low-value facts (founding year, headquarters, customer count), which inflates accuracy for both methods. It is also costly to label, and the vendor's own definition of a "data point" is unknown.
- The fair version is a field set fixed **before** running either method, ideally every numeric data point in the CIM. Report it alongside a separate high-value subset (financials and KPIs), each with its interval.